In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
    
GOOGLE_MODEL = "gemini-3.1-flash-lite"

llm = ChatGoogleGenerativeAI(
    model=GOOGLE_MODEL, 
    temperature=0,
)

In [3]:
from pydantic import BaseModel, Field

class StudentInfo(BaseModel):
    """자기소개에서 추출한 학생 정보"""
    name: str = Field(description="학생의 이름")
    age: int = Field(description="학생의 나이")
    major: str = Field(description="학생의 전공")
    hobbies: list[str] = Field(description="학생의 취미 목록")
    goal: str = Field(description="학생의 목표")

In [4]:
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=StudentInfo)
format_instructions = parser.get_format_instructions()

In [5]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 자기소개를 입력받아 학생 정보를 추출하는 AI입니다. 글에 있는 내용만 사용해 정보를 추출하세요.\n{format_instructions}"),
    ("user", "자기소개: {introduction}"),
]).partial(format_instructions=format_instructions)

chain = prompt | llm | parser

In [7]:
introduction =  "안녕하세요! 저는 김민수이고 22살입니다. 컴퓨터공학을 전공하고 있어요. 취미로는 게임하기, 영화보기, 코딩을 좋아합니다. 앞으로 훌륭한 개발자가 되는 것이 목표입니다."
result = chain.invoke({"introduction": introduction})

print(result.model_dump_json(indent=4))

{
    "name": "김민수",
    "age": 22,
    "major": "컴퓨터공학",
    "hobbies": [
        "게임하기",
        "영화보기",
        "코딩"
    ],
    "goal": "훌륭한 개발자가 되는 것"
}
